# Análisis científico de resultados F1 (protocolo v11)

Experimento de alto riesgo (risk_scale=1.5, risk_level=high) en grids 8×8 y 16×16.


## 1. Importar librerías y definir rutas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

raw_dir = Path('../raw')
files = {
    'grid8': {
        'json': raw_dir / 'grid8_riskhigh_seed42_v11.json',
        'csv': raw_dir / 'grid8_riskhigh_seed42_v11_episodes.csv'
    },
    'grid16': {
        'json': raw_dir / 'grid16_riskhigh_seed42_v11.json',
        'csv': raw_dir / 'grid16_riskhigh_seed42_v11_episodes.csv'
    }
}
display(files)


## 2. Cargar datos


In [ ]:
# Cargar CSV
df_grid8 = pd.read_csv(files['grid8']['csv']) if files['grid8']['csv'].exists() else pd.DataFrame()
df_grid16 = pd.read_csv(files['grid16']['csv']) if files['grid16']['csv'].exists() else pd.DataFrame()
display(df_grid8.head())
display(df_grid16.head())

# Cargar JSON
def load_json(path):
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

data_grid8 = load_json(files['grid8']['json'])
data_grid16 = load_json(files['grid16']['json'])
list(data_grid8.keys())


## 3. Limpieza básica


In [ ]:
def clean_df(df, label):
    if df.empty:
        print(f"{label}: DataFrame vacío")
        return df
    print(f"{label}: shape={df.shape}")
    return df.dropna()

df_grid8_clean = clean_df(df_grid8, 'grid8')
df_grid16_clean = clean_df(df_grid16, 'grid16')


## 4. Visualización de métricas clave


In [ ]:
metrics = ['Recompensa', 'Flexibilidad', 'Robustez', 'RiskEffective_Avg', 'Surprise_Avg', 'PGF_Bruto_Avg', 'PGF_Costo_Avg']

def plot_metric(df, metric, grid_label):
    if df.empty or metric not in df.columns:
        print(f"{grid_label}: métrica {metric} no disponible")
        return
    plt.figure(figsize=(8,5))
    sns.boxplot(x='Agente', y=metric, data=df)
    plt.title(f'{metric} por agente ({grid_label})')
    plt.show()

for metric in metrics:
    plot_metric(df_grid8_clean, metric, 'grid8')
    plot_metric(df_grid16_clean, metric, 'grid16')


## 5. Resumen estadístico por agente


In [ ]:
def resumen_agente(df, label):
    if df.empty:
        print(f"{label}: sin datos")
        return
    print(f"--- {label} ---")
    print(df.groupby('Agente').agg({
        'Recompensa': ['mean', 'std'],
        'Flexibilidad': 'mean',
        'Robustez': 'mean',
        'RiskEffective_Avg': 'mean',
        'Surprise_Avg': 'mean',
        'PGF_Bruto_Avg': 'mean',
        'PGF_Costo_Avg': 'mean'
    }))

resumen_agente(df_grid8_clean, 'grid8')
resumen_agente(df_grid16_clean, 'grid16')


## 6. Exportar tabla resumen


In [ ]:
summary_rows = []
for df, label in [(df_grid8_clean, 'grid8'), (df_grid16_clean, 'grid16')]:
    if df.empty:
        continue
    agg = df.groupby('Agente').agg({
        'Recompensa': ['mean', 'std'],
        'Flexibilidad': 'mean',
        'Robustez': 'mean',
        'RiskEffective_Avg': 'mean',
        'Surprise_Avg': 'mean',
        'PGF_Bruto_Avg': 'mean',
        'PGF_Costo_Avg': 'mean',
    })
    agg.columns = ['_'.join([str(c) for c in cols if c]) for cols in agg.columns.values]
    agg = agg.reset_index()
    agg.insert(0, 'Grid', label)
    summary_rows.append(agg)

if summary_rows:
    summary_df = pd.concat(summary_rows, ignore_index=True)
    out_path = Path('resumen_F1_v11_metricas.csv')
    summary_df.to_csv(out_path, index=False)
    print('Tabla resumen guardada en:', out_path)
    display(summary_df)
else:
    print('No hay datos para exportar.')
